# LeanCore на Kaggle (GPU T4×2 / P100)

Полный цикл: **данные → обучение fp32 → QAT-KL → экспорт LCW2** для нашего C-движка/сервиса.
Архитектура верифицирована против numpy/C-реализации в репозитории (gradcheck + численные сверки).

## Бюджеты Kaggle (бесплатный тир)
- **T4×2**: ~30 ч GPU-квоты/неделю; сессия до ~12 ч
- **P100**: ~30 ч/неделю; сессия до ~12 ч
- Хранилище `/kaggle/working` — артефакты скачиваются как Output ноутбука

## Сколько данных и времени брать (наш опыт):
| Корпус | Токенов | Шагов (B=64) | Эпох | Рекомендация |
|---|---|---|---|---|
| Shakespeare+Milton (старый) | 1.07M | 1500–2000 | 3–4 | быстрый щуп, V=8000 |
| + Спенсер/Марло/Бэкон/Донн/KJV↯ | 1.84M | 2500–3500 | 3–4 | **стандарт**, V=16000 |
| полный (без cap KJV) | 2.86M | 3500–4500 | 3 | если 10+ ч |

На GPU шаг (B=64,T=96,D=192,L=4) ≈ 0.05–0.15 с ⇒ **4000 шагов ≈ 3–8 мин вычислений**; доминирует старт/данные/eval.


In [ ]:
# === 0. окружение ===
import os, sys, math, time, json, re, glob, subprocess, urllib.request, collections
import numpy as np
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
WORK = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()


In [ ]:
# === 1. данные: тянем наш же корпус из GitHub-репо (raw.githubusercontent на Kaggle открыт) ===
# Kaggle → Add Data НЕ требуется. Если интернет выключен (Settings→Internet), включи и перезапусти.
REPO = "https://raw.githubusercontent.com/Riyozaki/AIra/arena/01a0432d-aira/leancore"
HEAD = REPO
URLS = {
  'shakespeare-0.6.tar.gz': HEAD + '/data/pkg/shakespeare-0.6.tar.gz',
}
books = ['A_1962-0.txt','Paradise_58.txt','Spenser_15272-8.txt','Spenser_72698-0.txt',
         'The_30.txt','The_48688-0.txt','The_575.txt','The_779-8.txt']
for b in books: URLS[b] = HEAD + '/data/books_txt/' + b
os.makedirs(f'{WORK}/corpus', exist_ok=True)
for name, url in URLS.items():
    dst = f'{WORK}/corpus/{name}'
    if os.path.exists(dst): continue
    try:
        urllib.request.urlretrieve(url, dst)
        print('ok', name)
    except Exception as e:
        print('FAIL', name, e)
subprocess.run(['tar','xzf',f'{WORK}/corpus/shakespeare-0.6.tar.gz','-C',f'{WORK}/corpus'], check=True)
print(sorted(os.listdir(f'{WORK}/corpus'))[:5], '...')


In [ ]:
# === 2. prep: очистка → дедуп → word-токенайзер → train/val ===
CAP_PER_FILE = 700_000        # KJV не задавит микс (жанровый баланс)
V = 16000                     # 8000 — если короткая сессия
def clean(txt):
    m = re.search(r"\*\*\*\s*START OF[^*]*\*\*\*", txt)
    if m: txt = txt[m.end():]
    m = re.search(r"\*\*\*\s*END OF", txt)
    if m: txt = txt[:m.start()]
    txt = re.sub(r"\[.*?\]", " ", txt); txt = re.sub(r"\r", "", txt)
    return re.sub(r"\n{3,}", "\n\n", txt).strip()

files = sorted(glob.glob(f'{WORK}/corpus/shakespeare-0.6/shksprdata/texts/*_gut.txt')
             + [f'{WORK}/corpus/shakespeare-0.6/miltondata/texts/'+f for f in
                ('paradiseregained.txt','paradise_lost_(no_introduction)_gut.txt','areopagitica_gut.txt',
                 'comus_gut.txt','lallegro_il_penseroso_comus_and_lycidas_gut.txt','allegro.txt')]
             + sorted(glob.glob(f'{WORK}/corpus/*[0-9]*.txt')))
def load(files):
    parts, seen = [], set()
    for f in files:
        base = os.path.basename(f)
        if base in seen: continue
        seen.add(base)
        t = clean(open(f, encoding='utf-8', errors='ignore').read()[:CAP_PER_FILE])
        if len(t) > 3000: parts.append("\n\n"+t)
    lines, dup = [], set()
    for ln in "\n".join(parts).split("\n"):
        ln = ln.strip()
        if ln and ln.lower() not in dup: dup.add(ln.lower()); lines.append(ln)
    return lines
lines = load(files)
toks = re.findall(r"[a-z']+|[0-9]+|[^\s\w]", "\n".join(lines).lower())
cnt = collections.Counter(toks)
stoi = {"<pad>":0,"<bos>":1,"<unk>":2}
for w,_ in cnt.most_common(V-3): stoi[w] = len(stoi)
itos = {i:w for w,i in stoi.items()}
ids = np.array([stoi.get(t,2) for t in toks], dtype=np.int64)
n_val = int(len(ids)*0.06)
tr, va = ids[:-n_val], ids[-n_val:]
np.save(f'{WORK}/train.npy', tr.astype(np.uint16)); np.save(f'{WORK}/val.npy', va.astype(np.uint16))
json.dump({"stoi":stoi,"vocab":len(stoi)}, open(f'{WORK}/meta.json','w'))
print(f'files={len(files)} tokens={len(ids):,} OOV={(ids==2).mean():.2%} val={len(va):,}')


## 3. Модель + Muon (ядро, верифицировано против репо)
Слои: LayerNorm → EMA-миксер (канальный α=σ(th), замкнутая треугольная форма) → Wm → residual → LN → GELU-FFN → residual; блоки 1..3 с ADR-роутингом (top-k позиций по σ(x·rw+rb)); голова tied с эмбеддингом.


In [ ]:
# -*- coding: utf-8 -*-
"""leancore_torch.py — верный GPU-порт LeanCore (nano_lc/np_qat) под Kaggle.

Математика 1:1 с numpy/C версией (проверено gradcheck'ом там же):
  блок:  x += g·( mix + ffn2(gelu(ffn1(ln2(x+mix)))) )  при ADR-роутинге top-k по σ(x·rw+rb)
  миксер EMA:  a = σ(th);  h_t = a·h_{t−1} + (1−a)·x_t;  y = sc⊙h;  mix = y @ Wm
        (на GPU строится треугольная матрица M[t,k,d] = (1−a_d)a_d^{t−k} и einsum)
  head:  logits = H @ Eᵀ  (tied)
QAT-STE:  тернаризация meanabs per-out для fc1/fc2/Wm через detach-трюк.
Потоковый экспорт: npz → (в отдельной ячейке) LCW2-share8 для C-движка из репо.
"""

f32 = torch.float32


# ---------------------------------------------------------------- Muon (Keller Jordan)
@torch.no_grad()
def zeropower_via_newtonschulz5(G, steps=5):
    """Ортогонализация обновления, bf16-хвост допустим; форма (r,c)."""
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.to(torch.bfloat16)
    transposed = G.size(0) > G.size(1)
    if transposed: X = X.mT
    X = X / (X.norm() + 1e-7)
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * A @ A
        X = a * X + B @ X
    if transposed: X = X.mT
    return X


class MuonW:
    """Muon для 2D скрытых матриц (по белому списку имён), AdamW для остального."""
    def __init__(self, named_params, muon_keys=(".Wm", ".fc1", ".fc2"), lr=6e-4, mulr=0.02,
                 adam_betas=(0.85, 0.95), adam_eps=1e-8, wd=0.0):
        self.lr, self.mulr = lr, mulr
        self.mu, self.ad = [], []          # (name, param, kind)
        for n, p in named_params:
            kind = "muon" if (p.ndim == 2 and any(k in n for k in muon_keys)) else "adam"
            (self.mu if kind == "muon" else self.ad).append((n, p))
            if kind == "muon":
                st = {"buf": torch.zeros_like(p)}
            else:
                st = {"m": torch.zeros_like(p), "v": torch.zeros_like(p), "t": 0}
            self.state = getattr(self, "state", {})
            self.state[p] = st
        self.b1, self.b2, self.eps, self.wd = adam_betas[0], adam_betas[1], adam_eps, wd
        self.nesterov = True
        print(f"[MuonW] muon: {len(self.mu)} шт, adam: {len(self.ad)} шт", flush=True)

    @torch.no_grad()
    def step(self, lr=None):
        lr = self.lr if lr is None else lr
        for n, p in self.mu:
            g = p.grad
            st = self.state[p]; buf = st["buf"]
            buf.lerp_(g, 1 - 0.95)                     # momentum 0.95
            u = (g.lerp(buf, 0.95) if self.nesterov else buf)
            O = zeropower_via_newtonschulz5(u).to(p.dtype)
            scale = max(1.0, p.size(0) / p.size(1)) ** 0.5
            p.add_(O, alpha=-lr * self.mulr * scale)   # mulr=0.02 как в numpy-версии
        for n, p in self.ad:
            g = p.grad; st = self.state[p]
            st["t"] += 1; t = st["t"]
            st["m"].lerp_(g, 1 - self.b1); st["v"].mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            mh = st["m"] / (1 - self.b1 ** t); vh = st["v"] / (1 - self.b2 ** t)
            p.addcdiv_(mh, vh.sqrt().add_(self.eps), value=-lr)
            if self.wd: p.mul_(1 - lr * self.wd)


# ---------------------------------------------------------------- EMA-миксер (треугольная Σ-форма)
def ema_mix(X, th, sc):
    """X (B,T,D). h_t = a⊙h_{t−1} + (1−a)⊙x_t; y = sc⊙h. Возвращает (y)."""
    B, T, D = X.shape
    a = torch.sigmoid(th)                                   # (D,)
    tt = torch.arange(T, device=X.device)
    dd = (tt[:, None] - tt[None, :]).clamp(min=0).to(f32)   # t−k, k≤t
    alog = torch.log(a.clamp_min(1e-20))
    P = torch.exp(dd[:, :, None] * alog[None, None, :])     # a^{t−k}
    mask = (tt[:, None] >= tt[None, :]).to(f32)
    M = P * mask[:, :, None] * (1 - a)[None, None, :]       # h_t = Σ_k M·x_k
    H = torch.einsum('tkd,bkd->btd', M, X)
    return H * sc


class Block(nn.Module):
    def __init__(self, D, ff, routed=False):
        super().__init__()
        self.routed = routed
        self.ln1 = nn.LayerNorm(D); self.ln2 = nn.LayerNorm(D)
        self.th = nn.Parameter(torch.zeros(D)); self.sc = nn.Parameter(torch.ones(D))
        self.Wm = nn.Parameter(torch.empty(D, D)); nn.init.normal_(self.Wm, std=0.02)
        self.fc1 = nn.Parameter(torch.empty(D, ff)); nn.init.normal_(self.fc1, std=0.02)
        self.fc2 = nn.Parameter(torch.empty(ff, D)); nn.init.normal_(self.fc2, std=0.02)
        if routed:
            self.rw = nn.Parameter(torch.empty(D)); nn.init.normal_(self.rw, std=0.02)
            self.rb = nn.Parameter(torch.tensor(1.5))
        # LayerNorm как {w=1,b=0} по умолчанию — совпадает с numpy-инициализацией

    def arm(self, x):
        """x (B,k,D) → delta (B,k,D): ln1→EMA→Wm→+resid→ln2→FFN; вернуть mix+ffn_out."""
        ln1 = self.ln1(x)
        mix = ema_mix(ln1, self.th, self.sc) @ self.Wm      # (B,k,D)
        h2 = x + mix
        z = self.ln2(h2)
        o2 = F.gelu(z @ self.fc1, approximate='tanh') @ self.fc2
        return mix + o2

    def forward(self, x, kfrac=None):
        """x (B,T,D). Если routed: top-k позиций по σ(x·rw+rb), gate σ, остальные наследуют вход."""
        if not self.routed or kfrac is None:
            return x + self.arm(x)
        B, T, D = x.shape
        s = (x * self.rw).sum(-1) + self.rb                 # (B,T)
        k = max(1, int(round(kfrac * T)))
        idx = s.topk(k, dim=1).indices.sort(dim=1).values   # (B,k)
        xs = torch.gather(x, 1, idx[:, :, None].expand(B, k, D))
        gs = torch.sigmoid(torch.gather(s, 1, idx))         # (B,k)
        delta = self.arm(xs) * gs[:, :, None]
        return x.scatter_add(1, idx[:, :, None].expand(B, k, D), delta)


class LeanCore(nn.Module):
    def __init__(self, V, D=192, L=4, ff=576, T=96, adr_kf=0.5):
        super().__init__()
        self.V, self.D, self.L, self.T, self.adr_kf = V, D, L, T, adr_kf
        self.E = nn.Parameter(torch.empty(V, D)); nn.init.normal_(self.E, std=0.02)
        self.pos = nn.Parameter(torch.empty(T, D)); nn.init.normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([Block(D, ff, routed=(adr_kf is not None and i > 0))
                                     for i in range(L)])
        self.lnf = nn.LayerNorm(D)

    def forward(self, ids):
        B, T = ids.shape
        h = self.E[ids] + self.pos[:T][None]
        for b in self.blocks:
            h = b(h, kfrac=self.adr_kf)
        return self.lnf(h)

    def logits(self, h):
        return h @ self.E.t()                               # tied head


# ---------------------------------------------------------------- QAT: тернарный STE (np_qat-протокол)
TERN_KEYS = (".Wm", ".fc1", ".fc2")

def tern_meanabs(w):
    s = w.abs().mean(dim=-1, keepdim=True).clamp_min(1e-5)
    return (torch.clamp(torch.round(w / s), -1, 1) * s)

class QAT:
    """np_qat-протокол 1:1. Цикл:  quantize_() → fwd/bwd → opt.step() → absorb_()  (eval: apply_())."""
    def __init__(self, model):
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.ndim == 2 and any(k in n for k in TERN_KEYS):
                self.shadow[n] = p.detach().clone()

    @torch.no_grad()
    def quantize_(self, model):                             # p ← tern(shadow)  (точка измерения градиента)
        for n, p in model.named_parameters():
            if n in self.shadow:
                p.copy_(tern_meanabs(self.shadow[n]))

    @torch.no_grad()
    def absorb_(self, model):                               # shadow += (p_after_opt − tern(shadow)); p ← shadow
        for n, p in model.named_parameters():
            if n in self.shadow:
                self.shadow[n].add_(p - tern_meanabs(self.shadow[n]))
                p.copy_(self.shadow[n])

    @torch.no_grad()
    def apply_(self, model):                                # для eval/экспорта: p ← tern(shadow)
        for n, p in model.named_parameters():
            if n in self.shadow:
                p.copy_(tern_meanabs(self.shadow[n]))


# ---------------------------------------------------------------- экспорт в npz (совместимо с репо)
def to_numpy_sd(model):
    sd = model.state_dict()
    out = {}
    out["E"] = model.E.detach().cpu().numpy().astype("float32")
    out["pos"] = model.pos.detach().cpu().numpy().astype("float32")
    for i, b in enumerate(model.blocks):
        pr = f"b{i}."
        out[pr + "ln1g"] = b.ln1.weight.detach().cpu().numpy().astype("float32")
        out[pr + "ln1b"] = b.ln1.bias.detach().cpu().numpy().astype("float32")
        out[pr + "th"] = b.th.detach().cpu().numpy().astype("float32")
        out[pr + "sc"] = b.sc.detach().cpu().numpy().astype("float32")
        # наши W в torch хранятся как (in,out) — numpy-версия тоже (in,out): совпадает
        out[pr + "Wm"] = b.Wm.detach().cpu().numpy().astype("float32")
        out[pr + "ln2g"] = b.ln2.weight.detach().cpu().numpy().astype("float32")
        out[pr + "ln2b"] = b.ln2.bias.detach().cpu().numpy().astype("float32")
        out[pr + "fc1"] = b.fc1.detach().cpu().numpy().astype("float32")
        out[pr + "fc2"] = b.fc2.detach().cpu().numpy().astype("float32")
        if b.routed:
            out[pr + "rw"] = b.rw.detach().cpu().numpy().astype("float32")
            out[pr + "rb"] = b.rb.detach().cpu().numpy().astype("float32")
    out["lnfg"] = model.lnf.weight.detach().cpu().numpy().astype("float32")
    out["lnfb"] = model.lnf.bias.detach().cpu().numpy().astype("float32")
    return out

def save_npz(model, path):
    import numpy as np
    np.savez(path, **to_numpy_sd(model))
    print("saved", path, flush=True)


In [ ]:
# === 4. калибровка скорости + план сессии ===
TARGET_HOURS = 9            # сколько часов спалить (лимит сессии ~12 ч)
B, CTX = 64, 96             # GPU: батч 64 безопасен даже на 16GB (модель ~4M пар.)

m = LeanCore(V, L=4, adr_kf=0.5).to(DEVICE)
print('params:', f'{sum(p.numel() for p in m.parameters()):,}')

def tb(rr, arr=tr):
    st = rr.integers(0, len(arr)-CTX-1, size=B)
    return (torch.tensor(np.stack([arr[s:s+CTX] for s in st]), device=DEVICE),
            torch.tensor(np.stack([arr[s+1:s+CTX+1] for s in st]), device=DEVICE))

opt = MuonW(m.named_parameters(), lr=6e-4, mulr=0.02)
rng_loc = np.random.default_rng(41); t0 = time.time()
with torch.no_grad(): pass
for i in range(25):                       # warm + measure
    x, y = tb(rng_loc)
    m.zero_grad(set_to_none=True)
    loss = torch.nn.functional.cross_entropy(m.logits(m(x)).reshape(-1,V), y.reshape(-1))
    loss.backward(); opt.step()
if DEVICE=='cuda': torch.cuda.synchronize()
dt = time.time()-t0; tps = 25*B*CTX/dt
print(f'{tps:.0f} tok/s (шаг {dt/25*1000:.0f} мс)')
steps = int(TARGET_HOURS*3600*tps/(B*CTX)*0.9)   # 10% — запас на eval/ckpt
print(f'план: {steps} шагов ≈ {steps*B*CTX/(tps)/3600:.1f} ч обучения (эпох: {steps*B*CTX/len(tr):.1f})')


In [ ]:
# === 5. обучение fp32 (cosine LR, warmup 60) ===
import torch.nn.functional as FCE
EVAL_EVERY = 250
opt = MuonW(m.named_parameters(), lr=6e-4, mulr=0.02)   # свежий оптимизатор
rng = np.random.default_rng(42)
rrv = np.random.default_rng(1234)

def vloss(iters=4):
    m.eval(); tot = 0.0
    with torch.no_grad():
        for _ in range(iters):
            x,y = tb(rrv, arr=va); tot += float(FCE.cross_entropy(m.logits(m(x)).reshape(-1,V), y.reshape(-1)))
    m.train(); return tot/iters

hist=[]; t0=time.time()
for step in range(steps):
    lr = 6e-4 * min((step+1)/60, 0.1 + 0.45*(1+math.cos(math.pi*max(0,step-60)/max(1,steps-60))))
    x, y = tb(rng)
    m.zero_grad(set_to_none=True)
    loss = FCE.cross_entropy(m.logits(m(x)).reshape(-1,V), y.reshape(-1))
    loss.backward(); opt.step(lr)
    if step % EVAL_EVERY == 0 or step == steps-1:
        vl = vloss(); rec = dict(step=step, train=round(float(loss),4), val=round(vl,4),
                                 ppl=round(math.exp(vl),2), wall=round(time.time()-t0,1))
        hist.append(rec); print(rec, flush=True)
        save_npz(m, f'{WORK}/ckpt_fp32.npz')   # rolling checkpoint
print('FP32 done:', hist[-1])


In [ ]:
# === 6. QAT-KL: 400 шагов, KL к собственному fp32 (Gemma-рецепт) ===
# teacher = текущая fp32 модель (замороженная копия), T=2 — как np_qat --muon --kl
import copy
teacher = copy.deepcopy(m).eval()
for p in teacher.parameters(): p.requires_grad_(False)
qat = QAT(m)
opt2 = MuonW(m.named_parameters(), lr=1.5e-4, mulr=0.0075)
hist=[]; t0=time.time()
for step in range(400):
    x,y = tb(rng)
    qat.quantize_(m)                          # p ← tern(shadow)
    m.zero_grad(set_to_none=True)
    lg = m.logits(m(x))
    loss = FCE.cross_entropy(lg.reshape(-1,V), y.reshape(-1))
    with torch.no_grad():
        lt = teacher.logits(teacher(x))
        q = (lt/2.0).softmax(-1)
    ps = (lg/2.0).log_softmax(-1)
    kl = FCE.kl_div(ps, q, reduction='batchmean')      # KL(q||ps)
    (loss + kl).backward()
    opt2.step(1.5e-4)
    qat.absorb_(m)                            # shadow += Δ; p ← shadow
    if step % 50 == 0 or step == 399:
        qat.apply_(m); vl = vloss(); print(dict(step=step, val=round(vl,4), ppl=round(math.exp(vl),2)), flush=True)
qat.apply_(m); save_npz(m, f'{WORK}/ckpt_qat.npz')
print('QAT final val ppl:', round(math.exp(vloss()),2))


In [ ]:
# === 7. генерация (sanity) + экспорт LCW2 share8 для C-движка ===
m.eval()
prompt = "to be or not to be"
ids_p = torch.tensor([[stoi.get(t,2) for t in re.findall(r"[a-z']+|[^\s\w]", prompt.lower())]], device=DEVICE)
cur = ids_p.clone()
import torch as T_  # локально
for _ in range(60):
    with torch.no_grad():
        lg = m.logits(m(cur[:, -CTX:]))[:, -1]
    nxt = lg.argmax(-1, keepdim=True)
    cur = torch.cat([cur, nxt], 1)
text = ' '.join(itos[int(i)] for i in cur[0].tolist())
print(text[:400])
z = open(f'{WORK}/meta.json').read()
print('--- артефакты в /kaggle/working: ckpt_fp32.npz, ckpt_qat.npz, meta.json, train/val.npy ---')


In [ ]:
# === 8. LCW2 (share8) — для lc_stream.c / lc_serve.py из репо ===
import numpy as np, struct
d = dict(np.load(f'{WORK}/ckpt_qat.npz'))
D = int(d['E'].shape[1]); Vv = int(d['E'].shape[0]); T = int(d['pos'].shape[0])
L = max(int(k[1]) for k in d if k.startswith('b') and '.' in k)+1
recs = []
add = lambda n,a,dt: recs.append((n.encode(), dt, np.ascontiguousarray(a)))
def tern_outscale(W):
    s = np.abs(W).max(0).clip(1e-5)
    q = np.clip(np.rint(W/s[None,:]),-1,1).astype(np.int8)
    return np.ascontiguousarray(q.T), s.astype(np.float16)
# shared int8-E/head: E per-row int8 + f16-скейлы (голова читает те же строки)
Ef = d['E'].astype(np.float32)
es = (np.abs(Ef).max(1).clip(1e-8)/127.0).astype(np.float16)
qE = np.clip(np.rint(Ef / es.astype(np.float32)[:,None]), -127, 127).astype(np.int8)
add('E', np.ascontiguousarray(qE), 2); add('E.s', es, 3)
add('E.qs', qE.astype(np.int32).sum(1).astype(np.int32), 0)
add('pos', d['pos'].astype(np.float16), 3)
for i in range(L):
    p = f'b{i}.'
    add(p+'ln1', d[p+'ln1g'].astype(np.float16),3); add(p+'ln1.bias', d[p+'ln1b'].astype(np.float16),3)
    add(p+'th', d[p+'th'].astype(np.float16),3);  add(p+'sc', d[p+'sc'].astype(np.float16),3)
    q,s = tern_outscale(d[p+'Wm'].astype(np.float32))
    add(p+'Wm', q, 2); add(p+'Wm.s', s, 3)
    add(p+'Wm.qs', q.astype(np.int32).sum(1).astype(np.int32), 0)  # NB: sum по входному каналу (q транспонирован)
    add(p+'ln2', d[p+'ln2g'].astype(np.float16),3); add(p+'ln2.bias', d[p+'ln2b'].astype(np.float16),3)
    for nm in ('fc1','fc2'):
        q,s = tern_outscale(d[p+nm].astype(np.float32))
        add(p+nm, q, 2); add(p+nm+'.s', s, 3); add(p+nm+'.qs', q.astype(np.int32).sum(1).astype(np.int32), 0)
    if p+'rw' in d:
        add(p+'rw', d[p+'rw'].astype(np.float16),3)
        add(p+'rb', np.asarray(d[p+'rb'],np.float32).reshape(1).astype(np.float16),3)
add('lnf', d['lnfg'].astype(np.float16),3); add('lnf.bias', d['lnfb'].astype(np.float16),3)
path = f'{WORK}/kaggle5.lcw2'
with open(path,'wb') as f:
    f.write(b'LCW2'); f.write(struct.pack('<5I', L, D, 0, Vv, T)); f.write(struct.pack('<f', 0.4))
    f.write(struct.pack('<I', len(recs)))
    for nm, dt, a in recs:
        f.write(struct.pack('<Q', len(nm))); f.write(nm); f.write(bytes([dt]))
        f.write(struct.pack('<I', a.ndim)); f.write(struct.pack(f'<{a.ndim}I', *a.shape)); f.write(a.tobytes())
import os
print(f'wrote {path}: {os.path.getsize(path):,} bytes, {len(recs)} recs')
print('Скачай Output → файлы ckpt_qat.npz + kaggle5.lcw2 + meta.json — кладутся в leancore/results/,')
print('дальше: ./lc_stream results/kaggle5.lcw2 ppl data/prep5k/val.npy  |  python3 lc_serve.py 8080 results/kaggle5.lcw2')
